# Lesson 12: Differentiation and Edge Detection

An edge is, informally, a place where intensity changes quickly. That's a statement about a *derivative*. This lesson builds up discrete image derivatives &mdash; Prewitt, Sobel, and Scharr operators &mdash; as convolution kernels (Lesson 10), uses them inside the classic **Canny** edge detector, and finishes by turning the resulting edge maps into actual shapes with the **Hough transform**.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## From derivatives to gradients

Treat the image as a function $I(x,y)$. Its gradient $\nabla I = (I_x, I_y)$ points in the direction of steepest intensity increase, where $I_x = \partial I/\partial x$ and $I_y = \partial I/\partial y$ are the **partial derivatives** of image intensity with respect to the horizontal and vertical directions &mdash; how fast the intensity changes as you move one pixel right, or one pixel down. Two numbers summarize the gradient at each pixel:

- **magnitude** $\|\nabla I\| = \sqrt{I_x^2 + I_y^2}$ &mdash; how sharp the change is (large at edges, near zero on flat regions)
- **direction** $\theta = \mathrm{atan2}(I_y, I_x)$ &mdash; which way intensity is increasing fastest (perpendicular to the edge)

$I_x$ and $I_y$ are themselves computed by convolving the image with small derivative kernels.

In [ ]:
def make_test_image(size=200):
    img = np.zeros((size, size), dtype=np.uint8)
    cv2.rectangle(img, (30, 30), (100, 100), 200, -1)
    cv2.circle(img, (140, 140), 40, 150, -1)
    pts = np.array([[150, 30], [190, 90], [110, 90]], dtype=np.int32)
    cv2.fillPoly(img, [pts], 25)  # low-contrast triangle, for the threshold demo later
    return img

img = make_test_image()
plt.imshow(img, cmap='gray')
plt.title('Test image (edges at several orientations)')
plt.axis('off')
plt.show()

## The naive derivative: just a difference

The simplest $I_x$ estimate is a 1D central-difference kernel $[-1,\ 0,\ 1]$. It works, but with no smoothing in the perpendicular direction, it's maximally sensitive to noise.

In [ ]:
simple_x = np.array([[-1, 0, 1]], dtype=np.float64)
simple_y = simple_x.T

gx = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, simple_x)
gy = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, simple_y)
magnitude = np.sqrt(gx**2 + gy**2)

plt.imshow(magnitude, cmap='gray')
plt.title('Gradient magnitude, simple [-1, 0, 1] kernel')
plt.axis('off')
plt.show()

## Prewitt, Sobel, and Scharr: derivative + smoothing

Real operators combine a difference in one direction with a *smoothing* average in the perpendicular direction &mdash; this is what makes them robust to noise. They differ only in how they weight that smoothing:

| Operator | $x$-kernel | Perpendicular weighting |
|---|---|---|
| **Prewitt** | $\begin{bmatrix}-1&0&1\\-1&0&1\\-1&0&1\end{bmatrix}$ | uniform: 1, 1, 1 |
| **Sobel** | $\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}$ | binomial: 1, 2, 1 |
| **Scharr** | $\begin{bmatrix}-3&0&3\\-10&0&10\\-3&0&3\end{bmatrix}$ | 3, 10, 3 |

Each $y$-kernel is just the transpose of its $x$-kernel. Sobel's binomial weights approximate a small Gaussian, which is why it's the most commonly used default. Scharr's weights were chosen (numerically optimized) specifically to make the *estimated gradient direction* as rotationally accurate as possible &mdash; i.e. as close to a true continuous derivative as an integer $3\times3$ kernel can get &mdash; which matters most in applications like optical flow that depend on precise gradient angles rather than just edge location.

In [ ]:
prewitt_x = np.array([[-1, 0, 1]] * 3, dtype=np.float64)
prewitt_y = prewitt_x.T

operators = {
    'Simple': (simple_x, simple_y),
    'Prewitt': (prewitt_x, prewitt_y),
    'Sobel': None,   # use cv2.Sobel directly below
    'Scharr': None,  # use cv2.Scharr directly below
}

img_f = img.astype(np.float64)
magnitudes = {}
magnitudes['Simple'] = magnitude
magnitudes['Prewitt'] = np.sqrt(cv2.filter2D(img_f, cv2.CV_64F, prewitt_x)**2 +
                                 cv2.filter2D(img_f, cv2.CV_64F, prewitt_y)**2)
magnitudes['Sobel'] = np.sqrt(cv2.Sobel(img_f, cv2.CV_64F, 1, 0, ksize=3)**2 +
                               cv2.Sobel(img_f, cv2.CV_64F, 0, 1, ksize=3)**2)
magnitudes['Scharr'] = np.sqrt(cv2.Scharr(img_f, cv2.CV_64F, 1, 0)**2 +
                                cv2.Scharr(img_f, cv2.CV_64F, 0, 1)**2)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, (name, mag) in zip(axes, magnitudes.items()):
    ax.imshow(mag, cmap='gray')
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

On a clean, low-noise image like this, the four results look nearly identical &mdash; the practical differences between them are subtle and show up mainly under noise or when precise sub-pixel angle matters, not as a visibly different edge map.

### Noise robustness: why the perpendicular smoothing matters

To make the benefit of that perpendicular smoothing concrete, we measure each kernel's response to pure noise on a flat (edge-free) region. To compare fairly, we first rescale each kernel so it has the *same gain* on an ideal step edge (i.e. divide by the sum of its positive weights) &mdash; otherwise a kernel with larger raw coefficients would look noisier just from its overall scale, not its shape.

In [ ]:
rng = np.random.default_rng(0)
flat_noisy = np.full((150, 150), 128.0) + rng.normal(0, 15, (150, 150))

kernels = {
    'Simple [-1,0,1]': simple_x,
    'Prewitt': prewitt_x,
    'Sobel': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64),
    'Scharr': np.array([[-3, 0, 3], [-10, 0, 10], [-3, 0, 3]], dtype=np.float64),
}

print(f'{"kernel":>18} {"noise std (gain-normalized)":>30}')
for name, k in kernels.items():
    gain = k[k > 0].sum()
    response = cv2.filter2D(flat_noisy, cv2.CV_64F, k / gain)
    print(f'{name:>18} {response.std():>30.2f}')

The plain difference kernel is noticeably noisier than any of the $3\times3$ operators &mdash; averaging over 3 rows (or columns) while differencing cuts down the noise response substantially, which is exactly the point of building smoothing into the derivative kernel.

## From gradients to edges: the Canny detector

Simply thresholding gradient magnitude gives thick, noisy edge blobs. The **Canny** edge detector (<a href="../references.html#canny-1986">Canny, 1986</a><span class="landmark-paper">&#9733;</span>) refines this with a multi-stage pipeline:

1. **Smooth** with a Gaussian, to suppress noise before differentiating.
2. **Compute gradients** (Sobel, internally) to get magnitude and direction at every pixel.
3. **Non-maximum suppression**: at each pixel, keep the gradient magnitude only if it's a local maximum *along the gradient direction* &mdash; this thins wide gradient ridges down to single-pixel-wide lines.
4. **Double thresholding + hysteresis**: pixels above a high threshold are definite edges; pixels below a low threshold are discarded; pixels in between are kept only if they connect to a definite edge. This links up weak-but-real edge segments while suppressing isolated noise responses.

In [ ]:
calvin = cv2.imread('../img/calvin.png', cv2.IMREAD_GRAYSCALE)

calvin_sobel_mag = np.sqrt(cv2.Sobel(calvin.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3)**2 +
                           cv2.Sobel(calvin.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3)**2)
naive_edges = (calvin_sobel_mag > 150).astype(np.uint8) * 255  # crude: just threshold |gradient|
canny_edges = cv2.Canny(calvin, threshold1=80, threshold2=160)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, im, title in zip(axes, [calvin, naive_edges, canny_edges],
                          ['Original', 'Naive: threshold |gradient|\n(thick, blobby)', 'Canny\n(thin, connected)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Image source: <a href="https://www.nga.gov/artworks/37890-john-calvin" target="_blank" rel="noopener">National Gallery of Art</a></p>

### Threshold sensitivity

Canny's two thresholds trade off completeness against noise. Too low, and noise gets picked up as spurious edges; too high, and real (but low-contrast) edges are missed.

In [ ]:
noisy_calvin = np.clip(calvin.astype(np.float64) + rng.normal(0, 8, calvin.shape), 0, 255).astype(np.uint8)

threshold_pairs = [(20, 60), (80, 160), (150, 220)]

fig, axes = plt.subplots(1, len(threshold_pairs), figsize=(10, 3.5))
for ax, (lo, hi) in zip(axes, threshold_pairs):
    edges = cv2.Canny(noisy_calvin, lo, hi)
    ax.imshow(edges, cmap='gray')
    ax.set_title(f'thresholds=({lo}, {hi})', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

With low thresholds on the noisy image, speckle noise survives as spurious edge fragments scattered everywhere, especially in the flat background. At the highest thresholds, faint detail (the window panes, the fine hatching that shades the curtain and robe) disappears completely, while the figure's strong dark outline and the books' edges survive &mdash; a real example of Canny's threshold trading off noise rejection against sensitivity to faint-but-genuine edges.

## From edges to lines: the Hough transform

Edge detection finds *where* intensity changes sharply, but it doesn't know that a scattered set of edge pixels forms a straight line. The **Hough transform** (<a href="../references.html#hough-1962">Hough, 1962</a>; <a href="../references.html#duda-hart-1972">Duda & Hart, 1972</a>) answers a different question: given a set of edge points, which ones are consistent with lying on a common line?

Parameterize a line not as $y=mx+b$ (which blows up for vertical lines) but by its distance from the origin and the angle of its normal:

$$\rho = x\cos\theta + y\sin\theta$$

For a *fixed* edge point $(x,y)$, this equation traces out a sinusoidal curve in $(\rho,\theta)$ space as $\theta$ varies &mdash; every $(\rho,\theta)$ pair on that curve describes a line through $(x,y)$. The key idea: if several edge points are **colinear**, their sinusoids all cross at the *same* $(\rho,\theta)$ &mdash; the parameters of the line they share.

In [ ]:
line_points = [(30, 270), (150, 150), (270, 30)]  # three points on the same line, x + y = 300

# the line's true (rho, theta): its normal direction is (1,1)/sqrt(2), i.e. theta=45 degrees,
# and rho is the line's distance from the origin along that normal
theta_true = np.radians(45)
rho_true = 300 * np.cos(theta_true)

thetas = np.linspace(0, np.pi, 500)
plt.figure(figsize=(6, 3.5))
for x, y in line_points:
    rho = x * np.cos(thetas) + y * np.sin(thetas)
    plt.plot(np.degrees(thetas), rho, label=f'({x},{y})')
plt.axvline(45, color='gray', linestyle='--', linewidth=1)
plt.scatter([np.degrees(theta_true)], [rho_true], color='black', zorder=5, marker='x', s=80,
            label=f'true line: rho={rho_true:.1f}, theta=45°')
plt.xlabel('theta (degrees)')
plt.ylabel('rho')
plt.legend(fontsize=8)
plt.title('Each point traces a sinusoid; colinear points cross at one (rho, theta)')
plt.show()

# confirm all three curves genuinely pass through that point, not just visually
for x, y in line_points:
    rho_at_true_theta = x * np.cos(theta_true) + y * np.sin(theta_true)
    print(f'point ({x:>3},{y:>3}): rho at theta=45 deg = {rho_at_true_theta:.2f}  (true rho = {rho_true:.2f})')

### The accumulator: voting for lines

In practice, every edge pixel votes for its entire sinusoid of possible $(\rho,\theta)$ lines, into a discretized 2D **accumulator** array. The bin with the most votes is the line most consistent with the data. We build this from scratch for a single synthetic line, and check it against `cv2.HoughLines`.

In [ ]:
line_img = np.zeros((300, 300), dtype=np.uint8)
cv2.line(line_img, (30, 270), (270, 30), 255, 3)  # the same x+y=300 line as above

ys, xs = np.nonzero(line_img)
theta_bins = np.linspace(0, np.pi, 180, endpoint=False)
max_rho = int(np.hypot(*line_img.shape))
accumulator = np.zeros((2 * max_rho, len(theta_bins)), dtype=np.int32)

for x, y in zip(xs, ys):
    rho_vals = (x * np.cos(theta_bins) + y * np.sin(theta_bins)).astype(int) + max_rho
    accumulator[rho_vals, np.arange(len(theta_bins))] += 1

peak_rho_idx, peak_theta_idx = np.unravel_index(np.argmax(accumulator), accumulator.shape)
manual_rho, manual_theta = peak_rho_idx - max_rho, theta_bins[peak_theta_idx]

cv_lines = cv2.HoughLines(line_img, 1, np.pi / 180, threshold=100)
cv_rho, cv_theta = cv_lines[0, 0]

print(f'true line parameters:    rho={rho_true:.1f}, theta={np.degrees(theta_true):.1f} deg')
print(f'manual accumulator peak: rho={manual_rho}, theta={np.degrees(manual_theta):.1f} deg')
print(f'cv2.HoughLines result:   rho={cv_rho:.0f}, theta={np.degrees(cv_theta):.1f} deg')

# recover two far-apart points on the detected line, to draw it back on the image
a, b = np.cos(cv_theta), np.sin(cv_theta)
x0, y0 = a * cv_rho, b * cv_rho
pt1 = (int(x0 + 1000 * (-b)), int(y0 + 1000 * a))
pt2 = (int(x0 - 1000 * (-b)), int(y0 - 1000 * a))
overlay = cv2.cvtColor(line_img, cv2.COLOR_GRAY2BGR)
cv2.line(overlay, pt1, pt2, (0, 0, 255), 3)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(line_img, cmap='gray')
axes[0].set_title('Input: single line', fontsize=10)
axes[0].axis('off')

# most bins get only 1-2 votes from individual points' sinusoids, while the peak bin gets one
# vote per point on the line (~240 here) -- on a linear color scale that peak washes out
# everything else, so we log-compress the counts to make the fainter sinusoid trails visible too
axes[1].imshow(np.log1p(accumulator), cmap='hot', aspect='auto',
               extent=[0, 180, -max_rho, max_rho], origin='lower')
axes[1].scatter([np.degrees(theta_true)], [rho_true], edgecolor='cyan', facecolor='none', s=150,
                 linewidth=1.5, label='true (rho, theta)')
axes[1].set_xlabel('theta (degrees)')
axes[1].set_ylabel('rho')
axes[1].legend(fontsize=7)
axes[1].set_title('Accumulator (log-scaled)', fontsize=10)

axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title('Recovered line (red),\noverlaid on input', fontsize=10)
axes[2].axis('off')
plt.tight_layout()
plt.show()

As you can see, the basic Hough algorithm returns an infinite line (red output) rather than a line segment (white input).

### `cv2.HoughLinesP`: the practical version

The probabilistic Hough transform additionally returns line *segment* endpoints (not just infinite lines), and is what's normally used in practice. We test it on a scene with two lines of known orientation, run through an actual Canny edge detector first (Hough transforms operate on edge maps, not raw images).

In [ ]:
shapes_with_lines = np.zeros((300, 300), dtype=np.uint8)
cv2.line(shapes_with_lines, (30, 270), (270, 30), 200, 2)   # true angle: -45 degrees
cv2.line(shapes_with_lines, (50, 50), (250, 50), 200, 2)    # true angle: 0 degrees
cv2.circle(shapes_with_lines, (220, 220), 30, 150, -1)      # a distractor shape, no straight edges

edges = cv2.Canny(shapes_with_lines, 80, 160)
segments = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=60, minLineLength=80, maxLineGap=5)

# color each segment by which of the two true lines (-45 or 0 degrees) it's closer to
# (colors given in BGR, since `vis` gets converted to RGB before display below)
color_diagonal, color_horizontal = (255, 0, 0), (80, 80, 255)  # red, blue

vis = cv2.cvtColor(shapes_with_lines, cv2.COLOR_GRAY2BGR)
print(f'{"segment":>20} {"angle (deg)":>12}')
for x1, y1, x2, y2 in segments.reshape(-1, 4):
    angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
    print(f'({x1},{y1})-({x2},{y2})'.rjust(20), f'{angle:>12.1f}')
    color = color_diagonal if abs(angle - (-45)) < abs(angle - 0) else color_horizontal
    cv2.line(vis, (x1, y1), (x2, y2), color, 2)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [shapes_with_lines, edges, vis],
                          ['Original', 'Canny edges', 'Detected segments\n(red = -45°, blue = 0°)']):
    ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Every detected segment lands almost exactly on one of the two true line angles (-45 and 0 degrees) &mdash; each line typically yields two or three near-duplicate detections rather than one, because a drawn line has finite width, so Canny finds an edge on *each* side of the stroke, a few pixels apart. Real Hough-based pipelines usually merge nearby detections in a post-processing step for exactly this reason. The circle &mdash; which has no straight edges at all &mdash; correctly produces no line detections whatsoever.

### Beyond lines: Hough circles

The same voting idea generalizes to any shape describable by a small number of parameters. A circle needs 3 (center $x$, center $y$, radius $r$), so `cv2.HoughCircles` accumulates votes in a 3D parameter space instead of 2D.

In [ ]:
circle_img = np.zeros((200, 200), dtype=np.uint8)
true_center, true_radius = (100, 100), 50
cv2.circle(circle_img, true_center, true_radius, 255, 1)

detected = cv2.HoughCircles(circle_img, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                             param1=50, param2=20, minRadius=30, maxRadius=70)
cx, cy, r = detected[0, 0]

print(f'true circle:      center={true_center}, radius={true_radius}')
print(f'detected circle:  center=({cx:.1f}, {cy:.1f}), radius={r:.1f}')

vis = cv2.cvtColor(circle_img, cv2.COLOR_GRAY2BGR)
cv2.circle(vis, (int(cx), int(cy)), int(r), (0, 0, 255), 1)
cv2.drawMarker(vis, (int(cx), int(cy)), (0, 0, 255), cv2.MARKER_CROSS, 10)
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('Detected circle (red) overlaid on the true one')
plt.axis('off')
plt.show()

### Peeking inside: the circle accumulator

`cv2.HoughCircles` doesn't expose its internal accumulator, but building one from scratch is the same voting idea as the line accumulator above, with one more dimension: a circle needs a center *and* a radius, so each edge pixel votes for every $(c_x, c_y, r)$ triple it could be consistent with, for every candidate radius $r$.

In [ ]:
circle_edges = cv2.Canny(circle_img, 50, 100)
edge_ys, edge_xs = np.nonzero(circle_edges)

H, W = circle_img.shape
radii = np.arange(30, 71)
vote_thetas = np.linspace(0, 2 * np.pi, 72, endpoint=False)  # angle around each candidate circle
cos_t, sin_t = np.cos(vote_thetas), np.sin(vote_thetas)

circle_accumulator = np.zeros((len(radii), H, W), dtype=np.int32)
for ri, r in enumerate(radii):
    # for this radius, every edge point votes for every center that's r away from it
    cand_cx = np.round(edge_xs[:, None] - r * cos_t[None, :]).astype(int)
    cand_cy = np.round(edge_ys[:, None] - r * sin_t[None, :]).astype(int)
    valid = (cand_cx >= 0) & (cand_cx < W) & (cand_cy >= 0) & (cand_cy < H)
    np.add.at(circle_accumulator[ri], (cand_cy[valid], cand_cx[valid]), 1)

peak_r_idx, peak_cy, peak_cx = np.unravel_index(np.argmax(circle_accumulator), circle_accumulator.shape)
print(f'true circle:       center={true_center}, radius={true_radius}')
print(f'accumulator peak:  center=({peak_cx}, {peak_cy}), radius={radii[peak_r_idx]}')

radius_vs_cx = circle_accumulator.max(axis=1)  # max over cy
radius_vs_cy = circle_accumulator.max(axis=2)  # max over cx

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(radius_vs_cx, aspect='auto', cmap='hot', extent=[0, W, radii[-1], radii[0]])
axes[0].set_xlabel('cx'); axes[0].set_ylabel('radius')
axes[0].set_title('radius vs. cx (max over cy)', fontsize=10)
axes[1].imshow(radius_vs_cy, aspect='auto', cmap='hot', extent=[0, H, radii[-1], radii[0]])
axes[1].set_xlabel('cy'); axes[1].set_ylabel('radius')
axes[1].set_title('radius vs. cy (max over cx)', fontsize=10)
plt.tight_layout()
plt.show()

To avoid the difficulty of visualizing a 3D array, we instead take 2D slices: the maximum vote count over $c_y$ (giving radius vs. $c_x$) and over $c_x$ (giving radius vs. $c_y$). Each slice shows a bright X: every edge point's votes trace two diagonal lines as the candidate radius grows (one for centers to its left, one to its right), and all of them cross at the true center and radius, which is exactly the brightest point in each plot.

### Exercise

1. Increase the noise standard deviation in `flat_noisy` and re-run the noise-robustness comparison. Does the relative ordering of the four kernels change?
2. `cv2.Canny` accepts an `apertureSize` parameter for its internal Sobel step (default 3). Try `apertureSize=5` and compare the result to the default on the noisy image.
3. Canny's hysteresis step needs a *connected* path of above-low-threshold pixels between a weak edge and a strong one. Construct a small binary example (by hand, as a NumPy array) where a real edge is broken into two segments with a 1-pixel gap, and confirm that hysteresis fails to link them even though a human viewer would clearly see one edge.
4. Lower `cv2.HoughLinesP`'s `threshold` argument well below 60 on `shapes_with_lines`. Do you start getting spurious short segments from noise in the circle's Canny boundary, even though it has no straight edges? What does raising `minLineLength` do to fix this?
5. Add a third line to `shapes_with_lines` at a shallow angle (e.g. from `(30, 200)` to `(270, 150)`), and predict its angle before checking `HoughLinesP`'s output against your prediction.
6. `cv2.HoughCircles`'s `param2` argument is roughly an accumulator vote threshold (like `HoughLines`'s `threshold`) &mdash; lower values report more (and more spurious) circles. Sweep `param2` from 30 down to 10 on a noisy version of `circle_img` (add Gaussian noise first) and observe when false-positive circles start appearing.